When there is no natural reference class, generate one with :class:`AAWindowSampler` and assemble it into a ``df_parts`` with ``get_df_parts_from_windows``. Each part can use its **own** generator, so the background carries part-specific physicochemical priors — here a coil-propensity JMD-N, an alpha-helix TMD, and a coil-propensity JMD-C. This method only consumes window sets; it does not sample sequences itself:

In [1]:
import pandas as pd
import aaanalysis as aa

df_seq = aa.load_dataset(name="DOM_GSEC", n=8)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq, list_parts=["jmd_n", "tmd", "jmd_c"])
n_ref = len(df_parts)

aws = aa.AAWindowSampler(random_state=0)
dict_parts = {
    "jmd_n": aws.sample_synthetic(df_seq=df_seq, n=n_ref, window_size=10, generator="coil"),
    "tmd":   aws.sample_synthetic(df_seq=df_seq, n=n_ref, window_size=20, generator="alpha_helix"),
    "jmd_c": aws.sample_synthetic(df_seq=df_seq, n=n_ref, window_size=10, generator="coil"),
}
df_parts_ref = aa.SequenceFeature.get_df_parts_from_windows(dict_parts)
df_parts_ref.head()

,jmd_n,tmd,jmd_c
REF0,QGCATWPSPY,NFAASWMQMWSATAREVMGK,QGCATWPSPY
REF1,TAVASEVPHL,ADPPNIYYQPQIDQLGLVWH,TAVASEVPHL
REF2,ADRQQKYYRQ,MHMHIVENCTSFVAHDKSEA,ADRQQKYYRQ
REF3,RKDRNHNWYI,IECMGPEWHCNWKWLKNYWK,RKDRNHNWYI
REF4,PIPIKWGQCT,RLLSIRQWDQWYAVYWDYVT,PIPIKWGQCT


Concatenate the reference parts with the real parts as the reference class (0) and run CPP (a smaller ``split_kws`` suits the short base parts):

In [2]:
df_all = pd.concat([df_parts, df_parts_ref])
labels = [1] * len(df_parts) + [0] * len(df_parts_ref)
split_kws = sf.get_split_kws(n_split_max=5, steps_pattern=[3, 4], n_min=2, n_max=3, len_max=8)
aa.CPP(df_parts=df_all, split_kws=split_kws).run(labels=labels, n_filter=5)[["feature", "abs_auc"]].head()

1. CPP creates 107238 features for 32 samples
1.1 Assigning scale values to parts
   |                         | 0.0%

   |........                 | 33.3%

   |................         | 66.7%

   |.........................| 100.0%


1.2 Streaming pre-filter stats (mask in stream)
   |                         | 0.0%

   |................         | 66.7%

   |.........................| 100.0%

   |.........................| 100.0%
2. CPP pre-filters 5361 features (5.0%) with highest 'abs_mean_dif' and 'max_std_test' <= 0.2 (kept=92755 of 107238)


3. CPP filtering algorithm
4. CPP returns df of 5 unique features with general information and statistics


,feature,abs_auc
0,"TMD-Segment(4,5)-NAKH920108",0.5
1,"TMD-PeriodicPattern(N,i+3/4,2)-CHAM830108",0.5
2,"TMD-PeriodicPattern(C,i+4/3,3)-FUKS010105",0.5
3,"TMD-PeriodicPattern(C,i+3/3,1)-CHOC760104",0.5
4,"TMD-Segment(4,5)-KOEH090104",0.5


**What can go wrong?** Window lists must align by position; differing counts warn and truncate to the shortest, and unknown part names raise:

In [3]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("always")
    df = aa.SequenceFeature.get_df_parts_from_windows({"jmd_n": ["AAAA", "CCCC", "DDDD"], "tmd": ["EEEEEE", "FFFFFF"]})
    print("truncated to", len(df), "rows")
try:
    aa.SequenceFeature.get_df_parts_from_windows({"not_a_part": ["AAAA", "CCCC"]})
except ValueError as e:
    print("ValueError:", e)

truncated to 2 rows
ValueError: 'dict_parts' key 'not_a_part' should be one of: ['tmd', 'tmd_e', 'tmd_n', 'tmd_c', 'jmd_n', 'jmd_c', 'ext_c', 'ext_n', 'tmd_jmd', 'jmd_n_tmd_n', 'tmd_c_jmd_c', 'ext_n_tmd_n', 'tmd_c_ext_c']


/Users/stephanbreimann/Programming/1Packages/aaanalysis/aaanalysis/feature_engineering/_sequence_feature.py:1233: RuntimeWarning: window counts differ across parts {'jmd_n': 3, 'tmd': 2}; truncating to 2.
  warnings.warn(f"window counts differ across parts {counts}; truncating to {n_windows}.",
